# CNN no Dataset MNIST

Comparação entre a **LightCNN** (customizada) e a **ResNet18** (pré-treinada) no dataset MNIST.

- **MNIST**: 60.000 imagens de treino e 10.000 de teste, 28x28 pixels, escala de cinza, 10 classes (dígitos 0–9).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights

from models import LightCNN
from utils import (
    get_device, train_model, plot_training_curves,
    plot_comparison_bar, get_predictions, plot_confusion_matrix,
    visualize_activations, print_summary_table,
)

device = get_device()
print(f"Dispositivo: {device}")

## 1. Carregamento e Preparação dos Dados

In [ ]:
BATCH_SIZE = 64
NUM_WORKERS = 0  # Windows-safe

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.MNIST(
    root='./data', train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

CLASSES = [str(i) for i in range(10)]
print(f"Treino: {len(train_dataset)} amostras | Teste: {len(test_dataset)} amostras")

## 2. Visualização de Amostras

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f'Label: {label}')
    ax.axis('off')
plt.suptitle('Amostras do MNIST', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Treinamento da LightCNN

In [ ]:
EPOCHS_CUSTOM = 10
LR = 1e-3

model_custom = LightCNN(in_channels=1, num_classes=10, img_size=28).to(device)
criterion = nn.CrossEntropyLoss()
optimizer_custom = optim.Adam(model_custom.parameters(), lr=LR)

print(f"Parâmetros da LightCNN: {sum(p.numel() for p in model_custom.parameters()):,}")

history_custom = train_model(
    model_custom, train_loader, test_loader, criterion, optimizer_custom,
    device, epochs=EPOCHS_CUSTOM, model_name="LightCNN"
)

In [ ]:
plot_training_curves(history_custom, title="LightCNN — MNIST")

## 4. Fine-tuning da ResNet18

In [ ]:
# Transforms para ResNet18: converter 1 canal -> 3 canais e resize para 32x32
transform_resnet = transforms.Compose([
    transforms.Resize(32),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize((0.1307, 0.1307, 0.1307), (0.3081, 0.3081, 0.3081)),
])

train_dataset_rn = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=transform_resnet
)
test_dataset_rn = torchvision.datasets.MNIST(
    root='./data', train=False, download=True, transform=transform_resnet
)

train_loader_rn = DataLoader(train_dataset_rn, batch_size=BATCH_SIZE, shuffle=True,
                             num_workers=NUM_WORKERS, pin_memory=True)
test_loader_rn = DataLoader(test_dataset_rn, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)

In [ ]:
EPOCHS_RESNET = 10

model_resnet = resnet18(weights=ResNet18_Weights.DEFAULT)

# Congelar todas as camadas exceto layer4 e fc
for name, param in model_resnet.named_parameters():
    if "layer4" not in name and "fc" not in name:
        param.requires_grad = False

# Adaptar classificador para 10 classes
model_resnet.fc = nn.Linear(model_resnet.fc.in_features, 10)
model_resnet = model_resnet.to(device)

trainable = sum(p.numel() for p in model_resnet.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_resnet.parameters())
print(f"ResNet18 — Total: {total:,} | Treináveis: {trainable:,}")

optimizer_resnet = optim.Adam(
    filter(lambda p: p.requires_grad, model_resnet.parameters()), lr=1e-3
)

history_resnet = train_model(
    model_resnet, train_loader_rn, test_loader_rn, criterion, optimizer_resnet,
    device, epochs=EPOCHS_RESNET, model_name="ResNet18"
)

In [ ]:
plot_training_curves(history_resnet, title="ResNet18 — MNIST")

## 5. Comparação de Resultados

In [ ]:
results = {"LightCNN": history_custom, "ResNet18": history_resnet}

print_summary_table(results)
plot_comparison_bar(results, title="Comparação de Acurácia — MNIST")

## 6. Matrizes de Confusão

In [ ]:
y_true_c, y_pred_c = get_predictions(model_custom, test_loader, device)
plot_confusion_matrix(y_true_c, y_pred_c, CLASSES, title="Matriz de Confusão — LightCNN (MNIST)")

y_true_r, y_pred_r = get_predictions(model_resnet, test_loader_rn, device)
plot_confusion_matrix(y_true_r, y_pred_r, CLASSES, title="Matriz de Confusão — ResNet18 (MNIST)")

## 7. Visualização das Ativações dos Kernels (DESAFIO)

In [ ]:
# Selecionar uma imagem de exemplo
sample_img, sample_label = test_dataset[0]
sample_img_batch = sample_img.unsqueeze(0)  # (1, 1, 28, 28)

plt.figure(figsize=(2, 2))
plt.imshow(sample_img.squeeze(), cmap='gray')
plt.title(f'Imagem de entrada — Label: {sample_label}')
plt.axis('off')
plt.show()

print("\nAtivações da LightCNN:")
visualize_activations(model_custom, sample_img_batch, device,
                      title_prefix="LightCNN MNIST — ")

## 8. Análise e Discussão

### Resultados Observados

Ambos os modelos alcançam acurácia muito alta no MNIST (>99%), o que é esperado dado que o MNIST é um dataset relativamente simples para redes convolucionais modernas.

### Por que a diferença é pequena?

O MNIST contém imagens de baixa resolução (28x28) em escala de cinza com variação limitada. Mesmo uma CNN simples com 3 camadas convolucionais consegue aprender os padrões necessários para classificar dígitos com alta precisão. A vantagem do pré-treinamento da ResNet18 no ImageNet é marginal aqui, pois os features de baixo nível aprendidos em imagens coloridas naturais não se transferem tão bem para dígitos manuscritos em preto e branco.

### Custo Computacional

A LightCNN, com ~300K parâmetros, treina significativamente mais rápido que a ResNet18 (~11M parâmetros), mesmo com a maioria das camadas congeladas. Para o MNIST, o custo adicional da ResNet18 não se justifica pelo ganho marginal de acurácia.

### Melhorias Possíveis

- **Learning rate scheduling** (ex: ReduceLROnPlateau) para convergência mais fina
- **Data augmentation** leve (rotação, translação) para melhorar generalização
- **Mais épocas** caso o modelo ainda não tenha convergido
- **Arquitetura mais profunda** para ganhos marginais adicionais